In [0]:
# CELL 1: Install & Verify Libraries
# -------------------------------------------------------------------------
# 1. Install the library
%pip install beautifulsoup4
 
# 2. Verify installation immediately
try:
    from bs4 import BeautifulSoup
    print("✅ BeautifulSoup4 is installed and working correctly.")
except ImportError:
    print("❌ Error: BeautifulSoup4 failed to install. Please re-run this cell.")

In [0]:
# CELL 2: The Master Ingestion Function
# -------------------------------------------------------------------------
import requests
from bs4 import BeautifulSoup
import os
from datetime import datetime, timedelta
 
# -------------------------------------------------------------------------
# 1. Azure Storage Config (PASTE YOUR INFO BELOW)
# -------------------------------------------------------------------------
storage_account_name = "youshallnotpass" # <-- PASTE Name here
storage_account_key = "dobbyhasnomaster" # <-- PASTE Key here
 
# Connect to ADLS
spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)
 
# Auto-create folders if missing
spark.conf.set("fs.azure.createRemoteFileSystemDuringInitialization", "true")
 
# -------------------------------------------------------------------------
# 2. The Reusable Ingestion Function
# -------------------------------------------------------------------------
def ingest_dataset(dataset_name, page_url, file_pattern):
    print(f"\n🚀 STARTING INGESTION: {dataset_name.upper()}")
 
    # Define Dynamic Paths (bronze/units, bronze/projects, etc.)
    base_path = f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/{dataset_name}"
    landing_zone = f"{base_path}/landing/"
    archive_zone = f"{base_path}/archive/"
 
    # A. Link Finder
    print(f" 🔍 Finding download link on page...")
    headers = {'User-Agent': 'Mozilla/5.0'}
    response = requests.get(page_url, headers=headers)
    soup = BeautifulSoup(response.content, 'html.parser')
 
    fresh_url = None
    for link in soup.find_all('a', href=True):
        if "download" in link['href'] and file_pattern in link['href']:
            fresh_url = f"https://www.dubaipulse.gov.ae{link['href']}" if link['href'].startswith("/") else link['href']
            print(f" ✅ Found link: {fresh_url}")
            break
 
    if not fresh_url: 
        raise Exception(f"❌ Link not found for {file_pattern}")
 
    # B. Archive Current File (if exists)
    try:
        dbutils.fs.ls(f"{landing_zone}current.csv")
        today = datetime.now().strftime("%Y-%m-%d")
        dbutils.fs.mv(f"{landing_zone}current.csv", f"{archive_zone}{today}.csv")
        print(f" 📦 Archived old file.")
    except: 
        pass
 
    # C. Download & Upload
    local_path = f"/tmp/{dataset_name}.csv"
    print(f" ⬇️ Downloading & Uploading...")
    with requests.get(fresh_url, headers=headers, stream=True) as r:
        r.raise_for_status()
        with open(local_path, 'wb') as f:
            for chunk in r.iter_content(chunk_size=10*1024*1024):
                f.write(chunk)
 
    dbutils.fs.cp(f"file:{local_path}", f"{landing_zone}current.csv")
    os.remove(local_path)
    print(f"✅ FINISHED: {dataset_name} is ready.")
 
print("Master Function Loaded.")

In [0]:
# CELL 3: Verify Manually Staged Data
# -------------------------------------------------------------------------
# Since the live web scraper is blocked by network timeouts, we bypass 
# the ingest_dataset function and directly verify our manual Azure uploads.

print("🔍 Checking staged files in ADLS Bronze Container...\n")

# 1. Verify Units Landing Zone
try:
    units_path = f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/units/landing/"
    units_files = dbutils.fs.ls(units_path)
    print("✅ Units Directory Found!")
    for f in units_files:
        print(f"   └── {f.name} ({f.size} bytes)")
except Exception as e:
    print(f"❌ Cannot access Units path: {e}")

print("-" * 50)

# 2. Verify Projects Landing Zone
try:
    projects_path = f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/projects/landing/"
    projects_files = dbutils.fs.ls(projects_path)
    print("✅ Projects Directory Found!")
    for f in projects_files:
        print(f"   └── {f.name} ({f.size} bytes)")
except Exception as e:
    print(f"❌ Cannot access Projects path: {e}")